In [18]:
import os
import sys
import logging
from pathlib import Path
import config
import importlib

importlib.reload(config)

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "config.py").exists():
    BASE_DIR = CURRENT_DIR
else:
    BASE_DIR = CURRENT_DIR / "rpi_system"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import config
import tensorflow as tf
from tensorflow.keras import layers, models

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("Trainer")

# Point to raw-img dataset
DATASET_DIR = config.PROJECT_ROOT / "Data" / "Testing" / "raw-img"

print("TensorFlow Version:", tf.__version__)
print("Dataset Directory :", DATASET_DIR)

TensorFlow Version: 2.20.0
Dataset Directory : C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\Data\Testing\raw-img


In [19]:
# Load Dataset (80% Train, 20% Val)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=config.IMAGE_SIZE,
    batch_size=config.BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=config.IMAGE_SIZE,
    batch_size=config.BATCH_SIZE,
    label_mode="categorical",
)

# Italian-to-English Translation Dictionary
TRANSLATE = {
    "cane": "Dog",
    "cavallo": "Horse",
    "elefante": "Elephant",
    "farfalla": "Butterfly",
    "gallina": "Chicken",
    "gatto": "Cat",
    "lion": "Lion",
    "mucca": "Cow",
    "pecora": "Sheep",
    "scoiattolo": "Squirrel",
}

raw_classes = train_ds.class_names
class_names = [TRANSLATE.get(name.lower(), name.capitalize()) for name in raw_classes]
num_classes = len(class_names)
logger.info(f"Detected {num_classes} classes: {class_names}")

# Save English labels to models/labels.txt
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.OUTPUT_LABELS, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")
logger.info(f"Saved English class labels to: {config.OUTPUT_LABELS}")

# Prefetch for GPU/CPU pipeline speed
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 24033 files belonging to 10 classes.
Using 19227 files for training.
Found 24033 files belonging to 10 classes.
Using 4806 files for validation.


23:10:09 [INFO] Detected 10 classes: ['Dog', 'Horse', 'Elephant', 'Butterfly', 'Chicken', 'Cat', 'Lion', 'Cow', 'Sheep', 'Squirrel']
23:10:09 [INFO] Saved English class labels to: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\rpi_system\models\labels.txt


In [20]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation")

preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(config.IMAGE_SIZE[0], config.IMAGE_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # Freeze ImageNet backbone

inputs = tf.keras.Input(shape=(config.IMAGE_SIZE[0], config.IMAGE_SIZE[1], 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="edge_ai_animal_detector")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "edge_ai_animal_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         5,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 944,890 (3.60 MB)

 Trainable params: 5,770 (22.54 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [21]:
logger.info(f"Training for up to {config.EPOCHS} epochs...")
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.EPOCHS,
    callbacks=callbacks,
)

val_loss, val_acc = model.evaluate(val_ds)
logger.info(f"Training Complete! Final Validation Accuracy: {val_acc:.2%}")

23:10:13 [INFO] Training for up to 30 epochs...


Epoch 1/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 141s 457ms/step - accuracy: 0.7353 - loss: 0.8223 - val_accuracy: 0.9076 - val_loss: 0.3250 - learning_rate: 0.0010
Epoch 2/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 137s 455ms/step - accuracy: 0.8635 - loss: 0.4306 - val_accuracy: 0.9186 - val_loss: 0.2581 - learning_rate: 0.0010
Epoch 3/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 136s 451ms/step - accuracy: 0.8753 - loss: 0.3834 - val_accuracy: 0.9222 - val_loss: 0.2402 - learning_rate: 0.0010
Epoch 4/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 134s 443ms/step - accuracy: 0.8828 - loss: 0.3576 - val_accuracy: 0.9274 - val_loss: 0.2222 - learning_rate: 0.0010
Epoch 5/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 136s 452ms/step - accuracy: 0.8849 - loss: 0.3468 - val_accuracy: 0.9299 - val_loss: 0.2137 - learning_rate: 0.0010
Epoch 6/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 136s 452ms/step - accuracy: 0.8934 - loss: 0.3269 - val_accuracy: 0.9318 - val_loss: 0.2106 - learning_rate: 0.0010
Epoch 7/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 136s 451ms/step - accura

23:44:36 [INFO] Training Complete! Final Validation Accuracy: 93.95%


In [22]:
# PHASE 2: FINE-TUNING (UNFREEZE UPPER CONVOLUTIONAL LAYERS)
logger.info("Starting Phase 2: Fine-Tuning...")

# Unfreeze the base model
base_model.trainable = True

# Keep early layers (basic edges/colors) frozen, adapt complex shape layers
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

logger.info(f"Unfrozen {len(base_model.layers) - fine_tune_at} / {len(base_model.layers)} layers for fine-tuning.")

# Recompile with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Fine-tune callbacks (patience=3)
fine_tune_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

# Resume training for another 10-15 epochs
fine_tune_epochs = 15
total_epochs = history.epoch[-1] + 1 + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=history.epoch[-1] + 1,
    epochs=total_epochs,
    callbacks=fine_tune_callbacks,
)

# Evaluate final fine-tuned accuracy
val_loss, val_acc = model.evaluate(val_ds)
logger.info(f"Fine-Tuning Complete! Final Out-of-Sample Accuracy: {val_acc:.2%}")


23:50:30 [INFO] Starting Phase 2: Fine-Tuning...
23:50:30 [INFO] Unfrozen 30 / 157 layers for fine-tuning.


Epoch 16/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 186s 595ms/step - accuracy: 0.8615 - loss: 0.4148 - val_accuracy: 0.9330 - val_loss: 0.1992 - learning_rate: 1.0000e-05
Epoch 17/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 184s 609ms/step - accuracy: 0.8805 - loss: 0.3604 - val_accuracy: 0.9332 - val_loss: 0.2008 - learning_rate: 1.0000e-05
Epoch 18/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 201s 669ms/step - accuracy: 0.8917 - loss: 0.3325 - val_accuracy: 0.9357 - val_loss: 0.1958 - learning_rate: 1.0000e-05
Epoch 19/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 197s 655ms/step - accuracy: 0.8933 - loss: 0.3225 - val_accuracy: 0.9374 - val_loss: 0.1929 - learning_rate: 1.0000e-05
Epoch 20/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 173s 576ms/step - accuracy: 0.8993 - loss: 0.3083 - val_accuracy: 0.9372 - val_loss: 0.1913 - learning_rate: 1.0000e-05
Epoch 21/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 174s 576ms/step - accuracy: 0.9046 - loss: 0.2945 - val_accuracy: 0.9386 - val_loss: 0.1901 - learning_rate: 1.0000e-05
Epoch 22/30
301/301 ━━━━━━━━━━━━━━

00:36:57 [INFO] Fine-Tuning Complete! Final Out-of-Sample Accuracy: 94.11%


In [23]:
# Convert and Save as Optimized TFLite Model
logger.info("Converting model to optimized .tflite for Raspberry Pi...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(config.OUTPUT_TFLITE, "wb") as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
logger.info(f"Saved TFLite model to: {config.OUTPUT_TFLITE} ({size_mb:.2f} MB)")


00:38:22 [INFO] Converting model to optimized .tflite for Raspberry Pi...
00:38:24 [INFO] Function `function` contains input name(s) resource with unsupported characters which will be renamed to edge_ai_animal_detector_1_dense_4_1_biasadd_readvariableop_resource in the SavedModel.
00:38:24 [INFO] Function `function` contains input name(s) resource with unsupported characters which will be renamed to edge_ai_animal_detector_1_dense_4_1_biasadd_readvariableop_resource in the SavedModel.


INFO:tensorflow:Assets written to: C:\Users\sabbu\AppData\Local\Temp\tmplj5pb_c8\assets


00:38:27 [INFO] Assets written to: C:\Users\sabbu\AppData\Local\Temp\tmplj5pb_c8\assets


Saved artifact at 'C:\Users\sabbu\AppData\Local\Temp\tmplj5pb_c8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_923')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2015895706832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895707792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895707600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895707216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895708368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895707024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895707984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895708176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895705872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2015895709328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  

00:38:32 [INFO] Saved TFLite model to: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\rpi_system\models\animal_classifier.tflite (1.07 MB)


In [24]:
import cv2
import numpy as np

interpreter = tf.lite.Interpreter(model_path=str(config.OUTPUT_TFLITE))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

with open(config.OUTPUT_LABELS, "r") as f:
    labels = [line.strip() for line in f if line.strip()]

# Test on a sample cat image from the dataset
test_img_path = next((DATASET_DIR / "gatto").glob("*.jpeg"))
print(f"Testing on: {test_img_path.name}")

img = cv2.imread(str(test_img_path))
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
resized = cv2.resize(rgb_img, (config.INPUT_WIDTH, config.INPUT_HEIGHT))
input_data = np.expand_dims(resized, axis=0).astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], input_data)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])[0]

top_idx = int(np.argmax(output))
print(f"Predicted Animal : {labels[top_idx]}")
print(f"Confidence Score : {output[top_idx] * 100:.2f}%")

Testing on: 1.jpeg
Predicted Animal : Cat
Confidence Score : 99.91%


C:\Users\sabbu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
